In [ ]:
!pip install openai-whisper torch
%pip install transformers
%pip install faiss-cpu
%pip install sentence-transformers

In [2]:
import sys
import numpy as np
import faiss

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("NumPy path:", np.__file__)
print("FAISS:", faiss.__version__)

ModuleNotFoundError: No module named 'faiss'

In [3]:
import sys
import os
import site
import numpy as np

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("NumPy location:", np.__file__)
print("User site enabled:", site.ENABLE_USER_SITE)
print("PYTHONNOUSERSITE:", os.environ.get("PYTHONNOUSERSITE"))
print("\nPython search paths:")

for path in sys.path:
    print(path)

Python: c:\Users\taten\AppData\Local\Python\pythoncore-3.14-64\python.exe
NumPy: 2.5.4
NumPy location: C:\Users\taten\AppData\Roaming\Python\Python314\site-packages\numpy\__init__.py
User site enabled: True
PYTHONNOUSERSITE: None

Python search paths:
c:\Users\taten\AppData\Local\Python\pythoncore-3.14-64\python314.zip
c:\Users\taten\AppData\Local\Python\pythoncore-3.14-64\DLLs
c:\Users\taten\AppData\Local\Python\pythoncore-3.14-64\Lib
c:\Users\taten\AppData\Local\Python\pythoncore-3.14-64

C:\Users\taten\AppData\Roaming\Python\Python314\site-packages
c:\Users\taten\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages


In [1]:
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
import numpy as np
import hashlib
import faiss

# Define path to your Beocat silver layer
silver_path = Path("./ieee_fraud/silver")
raw_path = Path("./ieee_fraud/bronze/raw")
out_path = Path("../ieee_fraud/silver/vector_chunks")
out_path.mkdir(parents=True, exist_ok=True)

embedding_model_name = "jinaai/jina-embeddings-v2-base-en"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embedder = SentenceTransformer(embedding_model_name)

# # Read the transaction table directly from your local Parquet file
# trans_df = pd.read_parquet(silver_path / "ieee_train_transaction.parquet")

# # Now you can inspect or filter it just like a table dataframe
# print(trans_df.head())

ModuleNotFoundError: No module named 'transformers'

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Load the LightGBM-derived top features whitelist once when the script starts
silver_path = Path("./ieee_fraud/silver")
whitelist_path = silver_path / "top_features_whitelist.json"

with open(whitelist_path, "r") as f:
    TOP_FEATURES = json.load(f)

def row_to_text(row):
    tx_id = row.get("TransactionID", "unknown")
    dt = row.get("TransactionDT", "unknown")
    fraud_val = row.get("isFraud")
    
    fraud_status = "unlabeled" if pd.isna(fraud_val) else ("fraudulent" if int(fraud_val) == 1 else "legitimate")
    
    parts = [
        f"Transaction ID {tx_id} recorded at time delta {dt}.",
        f"Classification: {fraud_status}."
    ]
    
    # Iterate exclusively through the top features from your JSON whitelist
    for col in TOP_FEATURES:
        # Check if the column exists in the row and is not null
        if col in row and pd.notna(row[col]):
            parts.append(f"{col} is {row[col]}.")
            
    return " ".join(parts)
  
def stable_faiss_id(chunk_id: str) -> int:
    h = hashlib.sha256(chunk_id.encode()).digest()
    return int.from_bytes(h[:8], byteorder="big", signed=True)

In [ ]:
import pyarrow.parquet as pq
silver_path = Path("./ieee_fraud/silver")
# Read from your cleaned Silver Parquet file instead of Bronze CSV
silver_file = silver_path / "ieee_train_merged.parquet"
chunk_size = 50000 
batch_size = 50000 
chunk_idx = 0
# Convert Path to string for PyArrow compatibility
parquet_file = pq.ParquetFile(str(silver_file))

print("Starting safe chunk processing from the Silver Parquet layer...")

# Iterate through the Parquet file in row-group batches
for batch in parquet_file.iter_batches(batch_size=batch_size):
    # Convert the pyarrow batch directly to a Pandas DataFrame chunk
    chunk = batch.to_pandas()
    
    chunk["text"] = chunk.apply(row_to_text, axis=1)
    texts = chunk["text"].tolist()
    
    vecs = embedder.encode(texts, batch_size=4, normalize_embeddings=True).astype("float32")
    
    chunk_records = []
    for i, (_, row) in enumerate(chunk.iterrows()):
        doc_id = str(row["TransactionID"])
        c_id = f"txn_{doc_id}_{chunk_idx:04d}"
        
        chunk_records.append({
            "chunk_id": c_id,
            "doc_id": doc_id,
            "token_count": len(tokenizer(row["text"], add_special_tokens=False)["input_ids"]),
            "text": row["text"],
            "isFraud": int(row["isFraud"]) if pd.notna(row.get("isFraud")) else None,
            "faiss_id": stable_faiss_id(c_id),
            "vector": vecs[i].tolist()
        })
    
    # Flush each chunk immediately to disk to protect RAM
    chunk_df = pd.DataFrame(chunk_records)
    chunk_df.to_parquet(out_path / f"vector_chunk_{chunk_idx}.parquet", index=False)
    
    print(f"Processed, embedded, and flushed Silver chunk {chunk_idx} ({len(chunk)} rows) to disk.")
    chunk_idx += 1

print("All Silver-layer chunks successfully processed and vectorized!")